# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a Croissant-described dataset using the `mlcroissant` library, referencing dataset entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
# Print basic metadata description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their fields, referencing entities by their `@id`.

In [ ]:
# Explore available record sets and their fields using @id
record_sets = metadata.recordSet
if record_sets is None or len(record_sets) == 0:
    # Some schema may use .hasPart for main RecordSets
    has_part = getattr(metadata, 'hasPart', [])
    record_sets = [x for x in has_part if getattr(x, '@type', None) == 'RecordSet']

print("Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id'] if isinstance(rs, dict) else rs.@id} ({rs['name'] if isinstance(rs, dict) and 'name' in rs else getattr(rs, 'name', '')})")
    # Explore fields
    fields = rs['field'] if isinstance(rs, dict) and 'field' in rs else getattr(rs, 'field', [])
    print("  Fields:")
    for f in fields:
        print(f"    - {f['@id'] if isinstance(f, dict) else f.@id} ({f['name'] if isinstance(f, dict) and 'name' in f else getattr(f, 'name', '')})")

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis. All entities referenced by their `@id`.

In [ ]:
# Select the available record sets by their @id
record_set_ids = []
record_set_names = {}
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else rs.@id
    record_set_ids.append(rs_id)
    record_set_names[rs_id] = rs['name'] if isinstance(rs, dict) and 'name' in rs else getattr(rs, 'name', rs_id)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Columns for RecordSet '{record_set_names[record_set_id]}' (@id={record_set_id}):")
    print(dataframes[record_set_id].columns.tolist())
    print(dataframes[record_set_id].head(2))

## 4. Exploratory Data Analysis (EDA)
Apply processing steps: filtering, normalization, and grouping on numeric fields. All entities referenced by `@id`.

In [ ]:
# Choose a RecordSet and numeric field by @id
# Example: suppose main survey RecordSet has @id 'https://api.app.sen.science/frontiers/7160411/mental-health-records'
# and it contains GAD-7 score column with @id 'gad7_score', group by 'gender'

# For demonstration, we search for a likely numeric field
selected_rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[selected_rs_id] if selected_rs_id else pd.DataFrame()

# Try to find numeric fields likely to be GAD-7, PHQ-9, PSQ scores
possible_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ['gad7', 'phq9', 'psq', 'score', 'age'])]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = df.columns[0] if len(df.columns) else None

print(f"Numeric field selected (@id): {numeric_field_id}")
threshold = 10
if numeric_field_id and numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id].apply(pd.to_numeric, errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce') -
        filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce').mean()
    ) / filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce').std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Try grouping by a categorical field (example: gender, village, education)
    group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'village', 'education', 'marital'])]
    group_field_id = group_candidates[0] if group_candidates else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (average {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions and relationships between numeric and categorical fields using Seaborn.

In [ ]:
# Visualization: e.g., boxplot of GAD-7 scores by gender
if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,6))
    sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("Insufficient field information for boxplot.")

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² mental health survey dataset from Kilifi County, referencing all entities by their `@id`. With `mlcroissant`, record sets and fields can be dynamically explored, filtered, normalized, and visualized, supporting further research and analysis.